# Split Datasets

Split dataset to Train/Validation/Test sets

## A. Overview

Random split to Train, Validation and Test sets

In [2]:
%pip install datasketch

Note: you may need to restart the kernel to use updated packages.


## B. Combine Datasets

In [3]:
from pathlib import Path
import csv
import os
import random
import json
import re
from collections import defaultdict

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from datasketch import MinHash, MinHashLSH

import configuration
from src import data_utils, setup

RANDS_STATE = 42
random.seed(RANDS_STATE)

dataset_path = Path('..') / 'data'

### B.1. Load Datasets

In [4]:
# disaster_frac = data_utils.get_data_disaster_fraction()
# disaster_filepath = dataset_path / 'disaster' / str(disaster_frac)

# df_disaster_informative = pd.read_csv(disaster_filepath / 'informative.csv')
# df_disaster_informative['subset'] = 'disaster'
# # df_disaster_humanitarian = pd.read_csv(disaster_filepath / 'humanitarian.csv')
# # df_disaster_humanitarian['subset'] = 'disaster'

In [5]:
extended_filepath = dataset_path / "extended"

df_weather = pd.read_csv(
    extended_filepath / "weather.csv"
)["tweet_text"].to_frame()
df_weather["informative"] = False
df_weather["subset"] = "weather"

df_out_topic = pd.read_csv(
    extended_filepath / "out_topic.csv"
)["tweet_text"].to_frame()
df_out_topic["informative"] = False
df_out_topic["subset"] = "out_topic"

### B.2. Combine Datasets

In [6]:
# df_informative = (
#     pd.concat([df_disaster_informative, df_weather, df_out_topic], ignore_index=True)
#     .sample(frac=1, random_state=setup.RANDOM_SEED)
#     .reset_index(drop=True)
# )
# df_informative.head()

## C. Filtering

### C.1. Single tokens

In [7]:
df_weather.shape
df_out_topic.shape

(10403525, 3)

In [12]:
def _normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def _character_shingles(text, shingle_size=5):
    if len(text) <= shingle_size:
        return {text} if text else set()
    return {text[i : i + shingle_size] for i in range(len(text) - shingle_size + 1)}


def _make_minhash(shingles, num_perm=64):
    mh = MinHash(num_perm=num_perm)
    if not shingles:
        mh.update(b"")
        return mh
    for shingle in shingles:
        mh.update(shingle.encode("utf-8", errors="ignore"))
    return mh


def filter_duplicates_minhash_lsh(
    df,
    text_column="tweet_text",
    similarity_threshold=0.8,
    num_perm=64,
    shingle_size=5,
    checkpoint_file="checkpoint_lsh.json",
):
    print(f"Original dataset size: {len(df)}")
    print("Running exact duplicate pre-pass...")

    normalized_texts = df[text_column].map(_normalize_text)
    seen_exact = set()
    indices_to_drop = set()
    kept_indices = []

    for idx, text in normalized_texts.items():
        if text in seen_exact:
            indices_to_drop.add(int(idx))
        else:
            seen_exact.add(text)
            kept_indices.append(int(idx))

    print(f"Removed {len(indices_to_drop)} exact duplicates before approximate matching.")
    print("Building MinHash LSH index and scanning for near-duplicates...")

    lsh = MinHashLSH(threshold=similarity_threshold, num_perm=num_perm)
    signatures = {}
    next_signature_id = 0

    for idx in kept_indices:
        if idx % 1000 == 0:
            print(f"Processing index {idx} / {len(df)}", end="\r")
        text = normalized_texts.loc[idx]
        shingles = _character_shingles(text, shingle_size=shingle_size)
        signature = _make_minhash(shingles, num_perm=num_perm)

        candidate_ids = lsh.query(signature)
        is_duplicate = False

        for candidate_id in candidate_ids:
            candidate_text = normalized_texts.loc[signatures[candidate_id]["index"]]
            candidate_shingles = signatures[candidate_id]["shingles"]
            if not shingles or not candidate_shingles:
                continue
            intersection = len(shingles & candidate_shingles)
            union = len(shingles | candidate_shingles)
            exact_jaccard = intersection / union if union else 0.0
            if exact_jaccard >= similarity_threshold:
                indices_to_drop.add(int(idx))
                is_duplicate = True
                break

        if not is_duplicate:
            signature_id = f"sig_{next_signature_id}"
            next_signature_id += 1
            signatures[signature_id] = {
                "index": int(idx),
                "shingles": shingles,
                "signature": signature,
            }
            lsh.insert(signature_id, signature)

    df_filtered = df.drop(index=list(indices_to_drop)).reset_index(drop=True)
    print(f"Final dataset size after near-duplicate removal: {len(df_filtered)}")
    return df_filtered

In [10]:
df_weather_processed = filter_duplicates_minhash_lsh(df_weather) # 41414

Original dataset size: 47518
Running exact duplicate pre-pass...
Removed 18 exact duplicates before approximate matching.
Building MinHash LSH index and scanning for near-duplicates...
Final dataset size after near-duplicate removal: 46441


In [ ]:
df_out_topic_processed = filter_duplicates_minhash_lsh(df_out_topic) # 41414

Original dataset size: 10403525
Running exact duplicate pre-pass...
Removed 21260 exact duplicates before approximate matching.
Building MinHash LSH index and scanning for near-duplicates...


In [ ]:
# test_df = df_out_topic.head(200).copy()
# test_processed = filter_duplicates_minhash_lsh(test_df)
# test_processed.shape

Original dataset size: 200
Running exact duplicate pre-pass...
Removed 0 exact duplicates before approximate matching.
Building MinHash LSH index and scanning for near-duplicates...
Final dataset size after near-duplicate removal: 195


(195, 3)